# Convolutional Neural Network

### Importing the libraries

In [17]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [18]:
tf.__version__

'2.21.0'

In [19]:
# Ubah parameter di sini untuk eksperimen
IMAGE_SIZE = (150, 150)  # Naik dari 128 untuk lebih banyak detail fitur
BATCH_SIZE = 32
EPOCHS = 60
FILTERS = 32        # Coba variasi: 16, 32, 64
KERNEL_SIZE = 3     # Coba variasi: 3, 5
L2_REG = 0.000005   # Dikurangi drastis — model tidak overfitting, regularisasi sebelumnya terlalu kuat
PRED_IMAGE_PATHS = ['single_prediction/cat_or_dog_1.jpg', 'single_prediction/cat_or_dog_2.jpg']
RANDOM_SEED = 42
tf.keras.utils.set_random_seed(RANDOM_SEED)

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [20]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=180,
    width_shift_range=0.15,
    height_shift_range=0.15,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)
training_set = train_datagen.flow_from_directory('training_set',
                                                 target_size=IMAGE_SIZE,
                                                 batch_size=BATCH_SIZE,
                                                 class_mode='binary')

Found 8000 images belonging to 2 classes.


### Preprocessing the Test set

In [21]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('test_set',
                                            target_size = IMAGE_SIZE,
                                            batch_size = BATCH_SIZE,
                                            class_mode = 'binary')

Found 2000 images belonging to 2 classes.


### Initialising the CNN

In [22]:
cnn = tf.keras.models.Sequential()

### Step 1 - Convolution

In [23]:
# Block 1: double conv (VGG-style) — 2 conv sebelum pooling, fitur lebih kaya
cnn.add(tf.keras.layers.Conv2D(
    filters=FILTERS, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG),
    input_shape=[IMAGE_SIZE[0], IMAGE_SIZE[1], 3]
))
cnn.add(tf.keras.layers.Conv2D(
    filters=FILTERS, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)
))
cnn.add(tf.keras.layers.BatchNormalization())

### Step 2 - Pooling

In [24]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [25]:
# Block 2: FILTERS*2, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 2, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 2, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Block 3: FILTERS*4, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Block 4: FILTERS*4, double conv
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Conv2D(filters=FILTERS * 4, kernel_size=KERNEL_SIZE, padding='same', activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.BatchNormalization())
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [26]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [27]:
cnn.add(tf.keras.layers.Dense(units=256, activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Dropout(0.4))
cnn.add(tf.keras.layers.Dense(units=128, activation='relu',
    kernel_regularizer=tf.keras.regularizers.l2(L2_REG)))
cnn.add(tf.keras.layers.Dropout(0.2))

### Step 5 - Output Layer

In [28]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [29]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [30]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

history = cnn.fit(x=training_set, validation_data=test_set, epochs=EPOCHS, callbacks=[early_stop, reduce_lr])

best_epoch = history.history['val_accuracy'].index(max(history.history['val_accuracy'])) + 1
train_acc_pct = history.history['accuracy'][best_epoch - 1] * 100
val_acc_pct = max(history.history['val_accuracy']) * 100
print(f'Best epoch: {best_epoch}')
print(f'Best train acc: {train_acc_pct:.2f}%')
print(f'Best val acc: {val_acc_pct:.2f}%')

Epoch 1/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 67s 258ms/step - accuracy: 0.5269 - loss: 0.9630 - val_accuracy: 0.4420 - val_loss: 0.9146 - learning_rate: 0.0010
Epoch 2/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 64s 257ms/step - accuracy: 0.5393 - loss: 0.7397 - val_accuracy: 0.5450 - val_loss: 0.7022 - learning_rate: 0.0010
Epoch 3/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 65s 258ms/step - accuracy: 0.5624 - loss: 0.7019 - val_accuracy: 0.5900 - val_loss: 0.6785 - learning_rate: 0.0010
Epoch 4/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 64s 257ms/step - accuracy: 0.5826 - loss: 0.6785 - val_accuracy: 0.5795 - val_loss: 0.6955 - learning_rate: 0.0010
Epoch 5/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 64s 257ms/step - accuracy: 0.5938 - loss: 0.6678 - val_accuracy: 0.5930 - val_loss: 0.6875 - learning_rate: 0.0010
Epoch 6/60
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.6048 - loss: 0.6655
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
250/250 ━━━━━━━━━━━━━━━━━━━━ 65s 258ms/step - accuracy: 0.611

## Part 4 - Making a single prediction

In [31]:
import numpy as np
from tensorflow.keras.preprocessing import image

class_indices = training_set.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

print('class_indices:', class_indices)
for pred_path in PRED_IMAGE_PATHS:
    test_image = image.load_img(pred_path, target_size = IMAGE_SIZE)
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis = 0)
    test_image = test_image / 255.0

    result = cnn.predict(test_image, verbose = 0)
    probability = float(result[0][0])
    predicted_index = 1 if probability >= 0.5 else 0
    predicted_label = idx_to_class[predicted_index]

    file_name = pred_path.split('/')[-1]
    print(f"{file_name}: prob={probability:.4f}, pred={predicted_label}")

class_indices: {'cats': 0, 'dogs': 1}
cat_or_dog_1.jpg: prob=0.9571, pred=dogs
cat_or_dog_2.jpg: prob=0.3464, pred=cats


In [32]:
# Dinonaktifkan: output prediksi sekarang sudah dicetak per file di cell sebelumnya
